---
title: "Authors"
format:
  html:
    code-fold: true
    code-summary: "Show code"
---

# Authors

This notebook explores author contribution patterns in the ClimateKG graph by tracing which chapters each author contributed to.

We use the live ClimateKG Wikibase instance and the authenticated SPARQL endpoint.

**Query UI**: https://climatekg.tibwiki.io/query/

**SPARQL endpoint**: https://climatekg.tibwiki.io/query/proxy/sparql

This notebook follows the same query-first workflow as the Report Structure notebook: validate the graph structure before moving to analysis.

In [1]:
import os
import pandas as pd
from SPARQLWrapper import JSON
from IPython.display import display, Markdown

from wikibase_auth import DEFAULT_SPARQL_ENDPOINT, DEFAULT_WIKIBASE_URL, build_sparql_client

os.environ.setdefault("CLIMATEKG_SPARQL_USERNAME", "ckg")
os.environ.setdefault("CLIMATEKG_SPARQL_PASSWORD", "fairdata")

SPARQL_ENDPOINT = os.getenv("CLIMATEKG_SPARQL_ENDPOINT", DEFAULT_SPARQL_ENDPOINT)
WIKIBASE_URL = os.getenv("CLIMATEKG_WIKIBASE_URL", DEFAULT_WIKIBASE_URL)

print(f"Endpoint: {SPARQL_ENDPOINT}")
print("Libraries imported successfully")

'ckg'

'fairdata'

Endpoint: https://climatekg.tibwiki.io/query/proxy/sparql
Libraries imported successfully


In [2]:
query = '''
PREFIX wd: <https://climatekg.tibwiki.io/entity/>
PREFIX wdt: <https://climatekg.tibwiki.io/prop/direct/>

SELECT ?author ?authorLabel ?chapter ?chapterLabel ?authorId
WHERE {
  ?author wdt:P20 ?authorId .
  ?author wdt:P27 ?chapter .
  ?chapter wdt:P1 wd:Q6 .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?authorLabel ?chapterLabel
LIMIT 200
'''

sparql = build_sparql_client(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
rows = results.get("results", {}).get("bindings", [])
df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in rows])

print(f"Rows returned: {len(df)}")
display(df.head(20))

Rows returned: 200


,chapter,author,authorId,authorLabel,chapterLabel
0,https://climatekg.tibwiki.io/entity/Q158,https://climatekg.tibwiki.io/entity/Q4750,AU0752,Adam Mohammed Sebbit,Mitigation Pathways Compatible with Long-term ...
1,https://climatekg.tibwiki.io/entity/Q141,https://climatekg.tibwiki.io/entity/Q4833,AU0835,Adelle Thomas,Cities and Settlements by the Sea
2,https://climatekg.tibwiki.io/entity/Q26,https://climatekg.tibwiki.io/entity/Q4833,AU0835,Adelle Thomas,Impacts of 1.5°C Global Warming on Natural and...
3,https://climatekg.tibwiki.io/entity/Q133,https://climatekg.tibwiki.io/entity/Q4833,AU0835,Adelle Thomas,Key Risks across Sectors and Regions
4,https://climatekg.tibwiki.io/entity/Q67,https://climatekg.tibwiki.io/entity/Q4592,AU0594,Aditi Mukherji,High Mountain Areas
5,https://climatekg.tibwiki.io/entity/Q192,https://climatekg.tibwiki.io/entity/Q4592,AU0594,Aditi Mukherji,Longer Report
6,https://climatekg.tibwiki.io/entity/Q190,https://climatekg.tibwiki.io/entity/Q4592,AU0594,Aditi Mukherji,Summary for Policymakers
7,https://climatekg.tibwiki.io/entity/Q116,https://climatekg.tibwiki.io/entity/Q4592,AU0594,Aditi Mukherji,Water
8,https://climatekg.tibwiki.io/entity/Q178,https://climatekg.tibwiki.io/entity/Q4007,AU0009,Adolf Acquaye,Industry
9,https://climatekg.tibwiki.io/entity/Q179,https://climatekg.tibwiki.io/entity/Q4484,AU0486,Adrian Leip,Cross-sectoral Perspectives


## Interpretation

Each row records an author linked to a chapter they contributed to. The `P20` property is a stable author identifier used to deduplicate repeated author entries, while `P27` captures the chapter contribution relationship.

This gives us a clean starting point for author-level analysis: we can count how many chapters each author contributed to, find repeated collaborators, and then extend the analysis to a more network-oriented view.